# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Transform Orders Data - String to JSON
1. Pre-process the JSON String to fix the Data Quality Issues
1. Transform JSON String to JSON Object
1. Write transformed data to the silver schema

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * 
  FROM gizmobox.bronze.v_orders;
```

In [ ]:
# Archived output only; no computation.

Archived table:
value
{"order_id": 1, "customer_id": 6973, "order_date": "2025-01-05", "transaction_timestamp": "2025-01-05 10:13:59", "total_amount": 499, "payment_method": "Bank Transfer", "items": [{"item_id": 8, "name": "Gaming Console", "category": "Electronics", "price": 499, "quantity": 1, "details": {"brand": "Sony", "color": "Blue"}}], "order_status": "Completed"}
{"order_id": 2, "customer_id": 3532, "order_date": 2025-01-19, "transaction_timestamp": "2025-01-19 00:05:13", "total_amount": 985, "payment_method": "PayPal", "items": [{"item_id": 4, "name": "Smartwatch", "category": "Electronics", "price": 299, "quantity": 2, "details": {"brand": "Canon", "color": "Gray"}}, {"item_id": 7, "name": "External Hard Drive", "category": "Electronics", "price": 129, "quantity": 3, "details": {"brand": "Dell", "color": "Blue"}}], "order_status": "Cancelled"}
{"order_id": 3, "customer_id": 3532, "order_date": "2025-01-08", "transaction_timestamp": "2025-01-08 23:11:00", "total_amount": 597

### 1. Pre-process the JSON String to fix the Data Quality Issues
[regexp_replace function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace)

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE OR REPLACE TEMP VIEW tv_orders_fixed
AS
SELECT value,
       regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value 
  FROM gizmobox.bronze.v_orders;
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):

### 2. Transform JSON String to JSON Object
- Function [schema_of_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/schema_of_json)
- Function [from_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json)

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT schema_of_json(fixed_value) AS schema,
       fixed_value
  FROM tv_orders_fixed
 LIMIT 1;
```

In [ ]:
# Archived output only; no computation.

Archived table:
schema	fixed_value
STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>	{"order_id": 1, "customer_id": 6973, "order_date": "2025-01-05", "transaction_timestamp": "2025-01-05 10:13:59", "total_amount": 499, "payment_method": "Bank Transfer", "items": [{"item_id": 8, "name": "Gaming Console", "category": "Electronics", "price": 499, "quantity": 1, "details": {"brand": "Sony", "color": "Blue"}}], "order_status": "Completed"}

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value,
       fixed_value
  FROM tv_orders_fixed;
```

In [ ]:
# Archived output only; no computation.

Archived table:
json_value	fixed_value
[
  6973,
  [
    [
      "Electronics",
      [
        "Sony",
        "Blue"
      ],
      8,
      "Gaming Console",
      499,
      1
    ]
  ],
  "2025-01-05",
  1,
  "Completed",
  "Bank Transfer",
  499,
  "2025-01-05 10:13:59"
]	{"order_id": 1, "customer_id": 6973, "order_date": "2025-01-05", "transaction_timestamp": "2025-01-05 10:13:59", "total_amount": 499, "payment_method": "Bank Transfer", "items": [{"item_id": 8, "name": "Gaming Console", "category": "Electronics", "price": 499, "quantity": 1, "details": {"brand": "Sony", "color": "Blue"}}], "order_status": "Completed"}

### 3. Write transformed data to the silver schema

Archived Databricks source — shown for reading, not Python execution.

```sql
DROP TABLE IF EXISTS gizmobox.silver.orders_json;
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders_json
AS
SELECT from_json(fixed_value, 
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') AS json_value
  FROM tv_orders_fixed;
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):
num_affected_rows	num_inserted_rows

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM gizmobox.silver.orders_json;
```

In [ ]:
# Archived output only; no computation.

Archived table:
json_value
[
  6973,
  [
    [
      "Electronics",
      [
        "Sony",
        "Blue"
      ],
      8,
      "Gaming Console",
      499,
      1
    ]
  ],
  "2025-01-05",
  1,
  "Completed",
  "Bank Transfer",
  499,
  "2025-01-05 10:13:59"
]
[
  3532,
  [
    [
      "Electronics",
      [
        "Canon",
        "Gray"
      ],
      4,
      "Smartwatch",
      299,
      2
    ],
    [
      "Electronics",
      [
        "Dell",
        "Blue"
      ],
      7,
      "External Hard Drive",
      129,
      3
    ]
  ],
  "2025-01-19",
  2,
  "Cancelled",
  "PayPal",
  985,
  "2025-01-19 00:05:13"
]
[
  3532,
  [
    [
      "Electronics",
      [
        "Apple",
        "White"
      ],
      3,
      "Wireless Headphones",
      199,
      3
    ]
  ],
  "2025-01-08",
  3,
  "Completed",
  "Bank Transfer",
  597,
  "2025-01-08 23:11:00"
]
[
  3532,
  [
    [
      "Electronics",
      [
        "Microsoft",
        "Black"
      ],
      2,
      "Laptop